# Abalone: baseline and model comparison
Rémi Barbier

A 60/20/20 split with seed 42 separates training, model selection and final evaluation. Encoders and scalers fit training rows only. MLflow records the experiment locally. No experiment files are committed.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
source = ROOT / "data/abalone.csv"
raw = pd.read_csv(source)
data = raw.rename(columns=lambda name: name.lower().replace(" ", "_"))
numeric = ["length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight"]
valid = data[numeric].gt(0).all(axis=1) & data.sex.isin(["M", "F", "I"])
valid &= data.rings.gt(0) & data[numeric + ["rings"]].notna().all(axis=1)
clean = data.loc[valid].copy()
development, test = train_test_split(clean, test_size=0.2, random_state=42)
train, validation = train_test_split(development, test_size=0.25, random_state=42)
features = ["sex"] + numeric


In [2]:
import logging
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*")
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

logging.getLogger("mlflow").setLevel(logging.ERROR)
mlflow.set_tracking_uri((ROOT / "mlruns").as_uri())
mlflow.set_experiment("abalone-notebook")

def pipeline(estimator):
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), numeric),
        ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["sex"]),
    ])
    return Pipeline([("preprocessing", preprocessing), ("model", estimator)])

def metrics(y, predictions):
    return {"rmse": float(np.sqrt(mean_squared_error(y, predictions))),
            "mae": mean_absolute_error(y, predictions), "r2": r2_score(y, predictions)}

candidates = {
    "mean_baseline": DummyRegressor(strategy="mean"),
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=1),
}
rows, fitted, runs = [], {}, {}
for name, estimator in candidates.items():
    with mlflow.start_run(run_name=name) as run:
        model = pipeline(estimator).fit(train[features], train.rings)
        result = metrics(validation.rings, model.predict(validation[features]))
        mlflow.log_params({"model": name, "seed": 42, "target": "rings", "data_sha256": hashlib.sha256(source.read_bytes()).hexdigest()})
        mlflow.log_metrics({f"validation_{key}": value for key, value in result.items()})
        rows.append({"model": name, **result})
        fitted[name], runs[name] = model, run.info.run_id
comparison = pd.DataFrame(rows).set_index("model").sort_values("rmse")
comparison.rename(columns={"rmse": "validation RMSE (rings)", "mae": "validation MAE (rings)", "r2": "validation R²"})

,validation RMSE (rings),validation MAE (rings),validation R²
model,,,
random_forest,2.154159,1.492244,0.545350
linear_regression,2.248790,1.576430,0.504527
mean_baseline,3.198233,2.358672,-0.002172


## Evaluate the selected model
Choose the lowest validation RMSE, then refit that model on the development set. The test set is used only for the selected model. Predictions remain continuous ring counts; age is an approximate conversion, not a separate target.

In [3]:
selected_name = comparison.index[0]
selected = pipeline(candidates[selected_name]).fit(development.sort_index()[features], development.sort_index().rings)
test_metrics = metrics(test.rings, selected.predict(test[features]))
with mlflow.start_run(run_id=runs[selected_name]):
    mlflow.log_metrics({f"test_{key}": value for key, value in test_metrics.items()})
result = {"selected_model": selected_name, "validation": comparison.reset_index().to_dict(orient="records"), "test": test_metrics}
import json
(ROOT / "reports").mkdir(exist_ok=True)
(ROOT / "reports/notebook_metrics.json").write_text(json.dumps(result, indent=2))
pd.Series({"model": selected_name, **test_metrics})

model    random_forest
rmse          2.214858
mae           1.540123
r2            0.570086
dtype: object

## Limits
The validation set guides a small model comparison rather than an extensive parameter search. Ring counts have a long upper tail, with fewer older examples. This holdout score does not provide individual uncertainty intervals or evidence of generalisation to new collection conditions.